In [ ]:
import numpy as np
from ngsolve.webgui import Draw
from ngsolve import CF
from fibermode.bragg import Bragg, BraggExact, BraggScalar, plotlogf
from scipy.optimize import newton

# Numerical computation of Bragg fiber modes

The last Bragg fiber related package in `fibermode` is the `Bragg` class. Similarly to the numerical solution for modes of a step-index fiber in `StepIndex`, the `Bragg` class solves the exact same Helmholtz eigenproblem from [Docs 1.2](./1_2_stepindex.ipynb) but instead on the concentric Bragg geometry. The same non-dimensionalization is used.

To refresh, we again utilize the `ModeSolver` class and FEAST to find the eigenpairs of interest within some defined search contour. We can utilize our previously calculated propagation constants from Docs 2.1 as a benchmark for our eigenvalues. The only difference in the geometry setup for `Bragg` is that, to determine leaky modes, there must be an `Outer` region which has the PML.

In [ ]:
C = Bragg(scale=5e-5, ts=[5e-5, 1e-5, 2e-5, 2e-5],
          mats=['air', 'glass', 'air', 'Outer'], ns=[1, 1.44, 1, 1],
          maxhs=[.2, .025, .08, .1], bcs=['r1', 'r2', 'R', 'OuterCircle'],
          wl=1.2e-6, ref=0,
          curve=8)

Because we utilize FEAST, we need to define a search contour for our desired modes. We can utilize the `sqrZfrom` method to get estimates of $Z$ values via the betas we calculated from the semi-analytical classes.

In [ ]:
A = BraggScalar(scale=5e-5,
                ts=[5e-5, 1e-5, 2e-5],
                ns=[1, 1.44, 1],
                mats=['air', 'glass', 'air'], 
                maxhs=[.2, .02, .08], 
                bcs=None, no_mesh=False,
                wl=1.2e-6, ref=0, curve=8)

k_low = A.k0 * A.ns[0] * A.scale
guess = np.array(.99995*k_low)
nu = 0
outer = 'h2'

beta1 = newton(A.determinant, guess, args=(nu, outer), tol = 1e-15)

Z1_true = C.sqrZfrom(beta1/A.scale)**.5  # method sqrZfrom gives Z^2 from beta (need to descale beta first)
Z1_true

With this `Z1_true`, we can look nearby to find our numerical propagation constants. This may require modifying radius, order, number of vectors sought, quadrature points, etc. to find the mode.

In [ ]:
center = Z1_true
radius = .1

p = 3
_, y, _, _, _, _ = C.leakymode(p, nspan=4, npts=4,
                                    rad=radius,
                                    ctr=center,
                                    alpha=5,
                                    niterations=5,
                                    nrestarts=0)

In [ ]:
for f in y:
    Draw(f, C.mesh)

We can do the same for our vector mode profiles. The method is called `leakyvecmode`.

In [ ]:
B = BraggExact(scale=5e-5,
                ts=[5e-5, 1e-5, 2e-5],
                ns=[1, 1.44, 1],
                mats=['air', 'glass', 'air'], 
                maxhs=[.2, .015, .04], 
                bcs=None, no_mesh=False,
                wl=1.2e-6, ref=0, curve=8)

k_low = B.k0 * B.ns[0] * B.scale
guess = np.array(.99995*k_low)
outer = 'h2'
nu = 1

beta2 = newton(B.determinant, guess, args=(nu, outer), tol = 1e-15)

Z2_true = C.sqrZfrom(beta2/A.scale)
Z2_true

In [ ]:
center = Z2_true
radius = .1
nspan = 4
npts = 4
p = 0

_, _, Es, phis, _ = C.leakyvecmodes(p=p, ctr=center, rad=radius,
                                       alpha=5,
                                       rhoinv=.9,
                                       quadrule='ellipse_trapez_shift',
                                       nspan=nspan, npts=npts,
                                       niterations=5, nrestarts=0,
                                       stop_tol=1e-9)

In [ ]:
for e in Es:
    Draw(e.real, C.mesh, vectors={'grid_size' : 100})

In [ ]:
for phi in phis:
    Draw(phi, C.mesh)